# Pesos, remuestreo y ensembles en ambas ramas

Se completa la respuesta al profesor antes de congelar modelos y evaluar el test revisado.

## Inicialización

Usamos la misma identidad de datos, código y protocolo en todos los cuadernos.

In [1]:
from pathlib import Path
import sys, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
while not (ROOT/'scripts/mlcore').is_dir():
    if ROOT == ROOT.parent: raise RuntimeError('No se encontro la raiz del proyecto')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'scripts'))
from mlcore.experiment import initialize, aligned
from mlcore.data import read_data, FOLDS, TEST
OUT = initialize()
print('Ejecucion verificada:', OUT.name)


Ejecucion verificada: 01afe756130bafc6


**Qué encontramos y cómo se usa.** La identidad evita mezclar artefactos de protocolos distintos. Los cálculos científicos viven en `scripts/mlcore`; estos cuadernos explican, ejecutan y muestran cada etapa.

## Predicciones previas para ensembles

Los pesos y el stacking usan temporadas anteriores; 2017/18 inicializa el primer fold. No hay un método de respaldo distinto en 2018/19.

In [2]:
from mlcore.experiment import feedback
if not (OUT/'oof_candidates.csv').exists(): feedback(OUT)
display(pd.read_csv(OUT/'ensemble_weights.csv'))
results=pd.read_csv(OUT/'model_comparison.csv')
display(results[results.candidate.str.contains('mean|weighted|stack')])

,branch,season,con/LR_REF,con/LR_consenso,con/LR,con/HGB,con/XGB,con/RF,con/MLP,con/SVM,sin/LR_historia,sin/LR_plantilla,sin/LR,sin/HGB,sin/XGB,sin/RF,sin/MLP,sin/SVM
0,con,1819,0.905569,0.000000e+00,0.057212,0.000000e+00,3.325451e-18,4.714924e-17,0.000000e+00,3.721971e-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,con,1920,0.982955,0.000000e+00,0.017045,4.647196e-18,1.455255e-18,2.410330e-17,0.000000e+00,0.000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,con,2021,0.940908,0.000000e+00,0.059092,0.000000e+00,3.371554e-17,0.000000e+00,0.000000e+00,1.041264e-17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,con,2122,0.905969,1.289997e-16,0.082208,0.000000e+00,0.000000e+00,0.000000e+00,1.537534e-17,1.182306e-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,con,2223,0.907429,5.199431e-17,0.092571,0.000000e+00,0.000000e+00,0.000000e+00,1.732445e-18,3.973748e-17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,con,2324,0.911339,1.087455e-16,0.084624,1.160096e-17,0.000000e+00,2.938188e-17,0.000000e+00,4.036712e-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,sin,1819,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.102404,0.316617,0.000000e+00,0.046288,0.000000,0.302293,0.232399,2.398605e-17
7,sin,1920,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.216317,0.213251,4.135519e-17,0.058662,0.110957,0.099997,0.300815,0.000000e+00
8,sin,2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.228733,0.213478,1.986334e-18,0.174907,0.061660,0.077087,0.244136,0.000000e+00
9,sin,2122,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.214492,0.201087,5.728969e-17,0.176239,0.139851,0.000000,0.267898,4.330150e-04


,candidate,n,log_loss,RPS,Brier,AUC_macro,accuracy,balanced_accuracy,ECE_masa,ECE_ancho,acierto_ganador_condicional,recall_empate
1,con/weighted,18184,0.965218,0.194984,0.572874,0.684354,0.538440,0.464340,0.008883,0.006541,0.716502,0.001770
3,con/mean,18184,0.967337,0.195547,0.574265,0.681889,0.538165,0.463130,0.010523,0.009753,0.716209,0.000000
11,con/stack,18184,0.974321,0.196413,0.579289,0.678916,0.527387,0.466817,0.014715,0.012826,0.713209,0.088515
13,sin/weighted,18184,0.977394,0.198726,0.581160,0.671953,0.531016,0.454889,0.015247,0.013844,0.706623,0.000000
14,sin/mean,18184,0.977599,0.198801,0.581351,0.671613,0.531401,0.454404,0.015104,0.013440,0.707135,0.000000
32,sin/stack,18184,0.985126,0.200105,0.586216,0.667227,0.523262,0.460810,0.015087,0.012382,0.700988,0.070812


**Qué encontramos / decisión.** Ambos ensembles se evalúan en las mismas seis temporadas y consumen configuraciones revisadas.

## Pesos y remuestreo solicitados

Se prueban LR y HGB de ambas ramas. El entrenamiento ponderado según el mercado se etiqueta explícitamente.

In [3]:
display(pd.read_csv(OUT/'market_guided_results.csv'))
print('Recalibracion posterior:',pd.read_csv(OUT/'oof_recalibration.csv').candidate.unique().tolist())

,candidate,n,log_loss,RPS,Brier,AUC_macro,accuracy,balanced_accuracy,ECE_masa,ECE_ancho,acierto_ganador_condicional,recall_empate
0,con/LR/none,18184,0.965199,0.194988,0.572867,0.684391,0.538715,0.464498,0.008538,0.006877,0.716795,0.001992
1,con/LR/undersample,18184,0.967604,0.195678,0.574515,0.684603,0.536571,0.466064,0.021827,0.020917,0.715404,0.019252
2,con/LR/balance,18184,0.967633,0.195685,0.574534,0.684575,0.536571,0.466032,0.021868,0.020970,0.715185,0.019031
3,con/LR/oversample,18184,0.967653,0.195693,0.574517,0.684533,0.537505,0.467099,0.021286,0.020279,0.715477,0.020580
4,con/HGB/none,18184,0.968530,0.195704,0.574816,0.680613,0.538495,0.464325,0.012702,0.009617,0.716795,0.000000
5,con/HGB/oversample,18184,0.972193,0.196934,0.577504,0.680675,0.538220,0.464851,0.028487,0.023556,0.716502,0.001770
6,con/HGB/undersample,18184,0.972513,0.197016,0.577736,0.680383,0.536296,0.463264,0.027924,0.025161,0.714160,0.005753
7,con/HGB/balance,18184,0.972651,0.197032,0.577825,0.680016,0.536131,0.463670,0.028419,0.024520,0.713794,0.005311
8,sin/HGB/none,18184,0.979474,0.199331,0.582538,0.669827,0.528982,0.451883,0.013270,0.011538,0.703915,0.000000
9,sin/LR/none,18184,0.980137,0.199215,0.582520,0.669522,0.529037,0.453824,0.016032,0.015056,0.704135,0.000885


Recalibracion posterior: ['con/LR/balance_recal', 'con/HGB/balance_recal', 'sin/LR/balance_recal', 'sin/HGB/balance_recal']


**Qué encontramos / decisión.** No se afirma que todas las variantes empeoren en cada temporada. El modelo sin cuotas puro no recibe estos pesos; son experimentos adicionales guiados por mercado.

## Congelar una selección por enfoque

Se elige menor log loss; dentro de 0,0005, se prefiere simplicidad. El test revisado aún no se genera en esta etapa.

In [4]:
from mlcore.experiment import freeze
frozen=freeze(OUT)
print(json.dumps(frozen['selected'],indent=2,ensure_ascii=False))

{
  "con": {
    "candidate": "con/LR_REF",
    "spec": {
      "branch": "con",
      "name": "LR_REF",
      "family": "LR",
      "columns": [
        "REF_logit",
        "REF_pD"
      ],
      "reduction": "all"
    },
    "parameter": {
      "C": 0.05
    },
    "validation_log_loss": 0.965199124283032
  },
  "sin": {
    "candidate": "sin/mean",
    "spec": null,
    "parameter": null,
    "validation_log_loss": 0.9775994901181446
  }
}


**Qué encontramos / decisión.** La selección queda vinculada a huellas de predicciones y protocolo. Cambiarla exige una ejecución nueva y no permite llamar prospectivo al test observado.